# 1. Data Preprocessor

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import math

class AISPreprocessor:
    def __init__(self, data_dir, input_seq_len=12, output_seq_len=1):
        self.data_dir = data_dir
        self.input_seq_len = input_seq_len
        self.output_seq_len = output_seq_len

        # 수동 설정된 범위로 MinMaxScaler 초기화
        # 정규화 범위 설정
        lat_range = (33.0, 38.0)
        lon_range = (124.0, 132.0)
        sog_range = (0.0, 100.0)
        cog_range = (0.0, 360.0)
        
        # MinMaxScaler 수동 설정
        self.scaler = MinMaxScaler()
        self.scaler.min_ = np.array([
            -lat_range[0] / (lat_range[1] - lat_range[0]),
            -lon_range[0] / (lon_range[1] - lon_range[0]),
            -sog_range[0] / (sog_range[1] - sog_range[0]),
            -cog_range[0] / (cog_range[1] - cog_range[0])
        ])
        self.scaler.scale_ = np.array([
            1 / (lat_range[1] - lat_range[0]),
            1 / (lon_range[1] - lon_range[0]),
            1 / (sog_range[1] - sog_range[0]),
            1 / (cog_range[1] - cog_range[0])
        ])
        self.scaler.feature_names_in_ = np.array(['위도', '경도', 'SOG', 'COG'])

    def load_and_preprocess(self):
        input_seqs = []
        output_seqs = []
        count = 1
        for file in os.listdir(self.data_dir):
            if file.endswith('.csv'):
                print(f"---------- {count}번째 파일 진행 중 ----------")
                count += 1
                df = pd.read_csv(os.path.join(self.data_dir, file), encoding='cp949')
                df = self._preprocess_single_file(df)
                in_seqs, out_seqs = self._extract_sequences(df)
                input_seqs.extend(in_seqs)
                output_seqs.extend(out_seqs)
        return np.array(input_seqs), np.array(output_seqs)

    def _preprocess_single_file(self, df):
        df = df[['일시', '위도', '경도', 'SOG', 'COG']].copy()
        df['일시'] = pd.to_datetime(df['일시'])
        df = df.sort_values('일시')
        df = df.dropna()
        df = df.set_index('일시').resample('5min').mean().interpolate()
        df = df.reset_index()
    
        # ------------------- 목적지 좌표 -------------------
        dest_lat = df['위도'].iloc[-1]
        dest_lon = df['경도'].iloc[-1]
        df['dest_lat'] = dest_lat
        df['dest_lon'] = dest_lon
    
        return df

    def _extract_sequences(self, df):
        input_seqs = []
        output_seqs = []
    
        total_len = self.input_seq_len + self.output_seq_len
        for i in range(0, len(df) - total_len, total_len // 4):
            input_window = df.iloc[i:i+self.input_seq_len]
            output_window = df.iloc[i+self.input_seq_len:i+total_len]
    
            # 입력: 위도, 경도, SOG, COG (정규화)
            input_scaled = self.scaler.transform(input_window[['위도', '경도', 'SOG', 'COG']])
    
            # 목적지 위도/경도
            dest_lat = input_window['dest_lat'].iloc[0]
            dest_lon = input_window['dest_lon'].iloc[0]
    
            # 목적지 위도/경도 정규화
            dest_scaled = self.scaler.transform([[dest_lat, dest_lon, 0, 0]])
            dest_lat_scaled = dest_scaled[0][0]
            dest_lon_scaled = dest_scaled[0][1]
            dest_scaled_coords = np.tile([dest_lat_scaled, dest_lon_scaled], (self.input_seq_len, 1))
    
            # Δlat, Δlon (정규화 X)
            delta_lat = (dest_lat - input_window['위도'].values).reshape(-1, 1)
            delta_lon = (dest_lon - input_window['경도'].values).reshape(-1, 1)

            # ----------
            # Vessel Trajectory Precdiction based on Attention Mechanisms"
            # Arxiv:2106.02002
            # 목적지 도달을 위한 경로 예측 시, 목적지까지의 거리와 도달 시간 추정치를 특성으로 추가하면 성능 향상됨을 보고
            # ----------
            # 거리
            distance = np.sqrt(delta_lat**2 + delta_lon**2).reshape(-1, 1)

            # 최종 입력 구성
            input_seq = np.hstack([
                input_scaled,            # 4
                dest_scaled_coords,      # 2
                distance,                # 1
            ])
            # 출력: 위도, 경도, SOG, COG (정규화)
            output_seq = self.scaler.transform(output_window[['위도', '경도', 'SOG', 'COG']])
    
            input_seqs.append(input_seq)
            output_seqs.append(output_seq)
    
        return input_seqs, output_seqs

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')
# 데이터 디렉토리 경로 설정 (예: routes 폴더 안에 여러 개의 csv가 있는 경우)
data_dir = './routes'
# 전처리 객체 생성
preprocessor = AISPreprocessor(data_dir)

# 데이터 로드 및 전처리 실행
input_seqs, output_seqs = preprocessor.load_and_preprocess()


# 출력 확인
print("Input sequences shape:", input_seqs.shape)   
print("Output sequences shape:", output_seqs.shape) 

# 예시 데이터 출력 
print("\nSample Input Sequence (첫 번째 샘플):")
print(input_seqs[5])

print("\nSample Output Sequence (첫 번째 샘플):")
print(output_seqs[5])

In [ ]:
import numpy as np
import pandas as pd
## 데이터 불러올때 참고 코드!
# 불러오기
input_df = pd.read_csv("input_seqs.csv", encoding='cp949', skiprows=0, header=None)
label_df = pd.read_csv("label_seqs.csv", encoding='cp949', skiprows=0, header=None)

# numpy 변환
input_seqs = input_df.to_numpy().reshape(-1, 12, 7)
output_seqs = label_df.to_numpy().reshape(-1, 1, 4)

# 예시 데이터 출력 (첫 샘플)
print("\nSample Input Sequence (첫 번째 샘플):")
print(input_seqs[5])

print("\nSample Output Sequence (첫 번째 샘플):")
print(output_seqs[5])

# 출력 확인
print("Input sequences shape:", input_seqs.shape)   # (num_samples, 43, 9)
print("Output sequences shape:", output_seqs.shape) # (num_samples, 1, 4)

# 2. Loss Function

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DestinationLoss(nn.Module):
    def __init__(self, weight_main=0.5):
        super().__init__()
        self.weight_main = weight_main  # 위치 vs 운동 손실 간의 가중치 설정

    def forward(self, y_pred, y_true):
        # y_pred, y_true: [B, 1, 4] (lat, lon, SOG, COG)
        y_pred = y_pred.squeeze(1)
        y_true = y_true.squeeze(1)

        pred_lat, pred_lon = y_pred[:, 0], y_pred[:, 1]
        true_lat, true_lon = y_true[:, 0], y_true[:, 1]
        pred_sog, pred_cog = y_pred[:, 2], y_pred[:, 3]
        true_sog, true_cog = y_true[:, 2], y_true[:, 3]

        # ------------------------------------------------------------
        # 위치 예측 손실 (위도/경도)
        # → 기본적으로 MSE 사용
        # 참고 논문: 
        #   - "TrajectoryNet: An Embedded GPS Trajectory Representation for Point-based Deep Learning"
        #     (https://arxiv.org/abs/2108.04909)
        #     → GPS 기반 trajectory 예측 시 위치 오차는 일반적으로 MSE 사용
        # ------------------------------------------------------------
        loc_loss = F.mse_loss(pred_lat, true_lat) + F.mse_loss(pred_lon, true_lon)

        # ------------------------------------------------------------
        # SOG (속력)에 대해 Huber Loss (Smooth L1)
        # → MSE보다 이상치(outlier)에 덜 민감
        # 참고 논문:
        #   - "Fast and Accurate Deep Network Learning by Exponential Linear Units (ELUs)"
        #     (https://arxiv.org/abs/1511.07289)
        #     → Regression task에서 Huber loss는 안정적인 수렴에 효과적
        #   - 자율주행 관련:
        #     "MultiNet: Real-time Joint Semantic Reasoning for Autonomous Driving"
        #     (https://arxiv.org/abs/1612.07695)
        # ------------------------------------------------------------
        sog_loss = F.smooth_l1_loss(pred_sog, true_sog)

        # ------------------------------------------------------------
        # COG (방위각) 손실 - 각도 오차 처리
        # → 단순 MSE는 0° vs 360° 문제로 잘못된 손실 계산 발생
        # 해결책: Circular Loss 적용
        # 참고 논문:
        #   - "On Learning to Simulate Navigation Paths with Geodesic-Consistent Features"
        #     (https://arxiv.org/abs/2011.08258)
        #     → 방향/각도 오차 계산에 있어서 periodic/circular loss 활용
        #   - "Object Detection on Spherical Images using an Equirectangular Grid"
        #     (https://arxiv.org/abs/1805.08999)
        #     → 방위각 예측에서 각도 차이를 주기적으로 계산
        # ------------------------------------------------------------
        cog_diff = torch.remainder(pred_cog - true_cog + 180, 360) - 180
        cog_loss = torch.mean(cog_diff**2)

        motion_loss = sog_loss + cog_loss

        # ------------------------------------------------------------
        # 최종 손실 계산
        # → 위치와 운동 손실을 가중합
        # 참고 구조:
        #   - "Vessel Trajectory Prediction using Attention Mechanisms"
        #     (https://arxiv.org/abs/2106.02002)
        #     → 위치, 속도, 방향 등 여러 loss component를 가중합하는 방식 사용
        # ------------------------------------------------------------
        total_loss = self.weight_main * loc_loss + (1 - self.weight_main) * motion_loss

        return total_loss

# 3. Transformer-based Regression Model

In [ ]:
import torch
import torch.nn as nn
# -------------------------------------------------------------------------------------
# [arXiv:1706.03762] Vaswani et al., "Attention Is All You Need"
# Learnable positional embedding (instead of sinusoidal encoding)
# -------------------------------------------------------------------------------------
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# -------------------------------------------------------------------------------------
# [arXiv:2304.14802] ResiDual: "Residual Connections are Natural Preconditioners"
# Dual LayerNorm (Pre + Post) in attention and FFN sublayers
# Improves gradient flow in deeper Transformers
# -------------------------------------------------------------------------------------
class ResiDualBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.ln1_pre = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ln1_post = nn.LayerNorm(d_model)

        self.ln2_pre = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),  # [arXiv:1606.08415] Gaussian Error Linear Unit
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.ln2_post = nn.LayerNorm(d_model)

    def forward(self, x):
        # Self-attention block
        x_norm = self.ln1_pre(x)
        attn_output, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.ln1_post(attn_output)

        # Feedforward block
        x_norm = self.ln2_pre(x)
        ff_output = self.ffn(x_norm)
        x = x + self.ln2_post(ff_output)
        return x
        
# -------------------------------------------------------------------------------------
# [arXiv:2111.11432] ConvNeXt / [arXiv:2201.03545] MetaFormer
# Residual FFN with GELU, acts as decoder bottleneck block
# -------------------------------------------------------------------------------------
class ResidualFFN(nn.Module):
    def __init__(self, dim, hidden_dim=None, dropout=0.1):
        super().__init__()
        hidden_dim = hidden_dim or dim * 4
        self.norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x):
        return x + self.ffn(self.norm(x))       
        
# -------------------------------------------------------------------------------------
# [arXiv:1706.03762] Transformer + [arXiv:2304.14802] ResiDual
# Time series regression with ResiDual attention blocks and residual decoder
# -------------------------------------------------------------------------------------
class TransformerPredictor(nn.Module):
    def __init__(self, input_size=7, output_size=4, d_model=128, nhead=8, num_layers=7, dim_feedforward=512, dropout=0.1, output_len=1, use_attention_pool=True):
        super().__init__()
        self.output_size = output_size
        self.output_len = output_len
        self.use_attention_pool = use_attention_pool

        # [arXiv:2010.11929] LayerNorm before projection
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_size),
            nn.SiLU(),
            nn.Linear(input_size, d_model//4),
            
            nn.LayerNorm(d_model//4),
            nn.SiLU(),
            nn.Linear(d_model//4, d_model//2),

            nn.LayerNorm(d_model//2),
            nn.SiLU(),
            nn.Linear(d_model//2, d_model)
        )
        self.pos_encoder = LearnablePositionalEncoding(d_model)

        # [arXiv:2304.14802] ResiDual encoder blocks
        self.encoder_layers = nn.ModuleList([
            ResiDualBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        # [arXiv:2005.12872] Attention-based pooling layer (like Set Transformer)
        if self.use_attention_pool:
            self.attn_pool = nn.Sequential(
                nn.Linear(d_model, d_model//2),
                nn.Tanh(),
                nn.Linear(d_model//2, 1)
            )

        # Decoder MLP with residuals
        self.decoder_input = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 256),
            nn.SiLU()  # [arXiv:1606.08415] Swish-1 variant
        )
        # Residual FFN decoder blocks
        self.res_blocks = nn.Sequential(
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
            ResidualFFN(256),
        )
        # Final projection to output space
        self.decoder_output = nn.Sequential(
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Linear(256, 128),

            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Linear(128, 64),

            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Linear(64, output_size * output_len)
        )

    def forward(self, x):
        # Encode
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        for layer in self.encoder_layers:
            x = layer(x)

        # Pooling
        if self.use_attention_pool:
            attn_scores = torch.softmax(self.attn_pool(x), dim=1)  # (B, S, 1)
            x_pooled = (attn_scores * x).sum(dim=1)  # (B, d_model)
        else:
            x_pooled = x.mean(dim=1)

        # Decode
        x_pooled = self.decoder_input(x_pooled)
        x_pooled = self.res_blocks(x_pooled)
        out = self.decoder_output(x_pooled)

        # Reshape
        return out.view(-1, self.output_len, self.output_size)

# 4. Train Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm import tqdm
import random

def train_transformer_model(model, train_data, val_data=None, num_epochs=30, batch_size=128, learning_rate=0.1*(1/256), device='cuda'):

    save_path = 'model_history/best_model.pth'
    model.to(device)

    x_train, y_train = train_data
    x_train_f, x_val, y_train_f, y_val = train_test_split(x_train, y_train, test_size=0.01, random_state=42)

    train_loader = DataLoader(TensorDataset(x_train_f, y_train_f), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size, shuffle=False)
    
    # 손실함수: 목적지 예측 오차 (사용자 정의 DestinationLoss)
    # 참고 논문: "Attention Is All You Need" (arXiv:1706.03762) 기반 Transformer 학습    
    criterion = DestinationLoss(weight_main=0.9)

    # 옵티마이저: AdamW (Weight Decay 분리) 사용
    # 참고 논문: "Decoupled Weight Decay Regularization" (arXiv:1711.05101)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.00025)

    # 학습률 스케줄러: Cosine Annealing with Warm Restarts
    # 참고 논문: "SGDR: Stochastic Gradient Descent with Warm Restarts" (arXiv:1608.03983)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs, T_mult=max(1, num_epochs//10))

    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        torch.cuda.empty_cache()
        model.train()
        total_loss = 0.0

        # Teacher Forcing 비율 점진적 감소
        # 참고 논문: "Scheduled Sampling" (arXiv:1506.03099)
        teacher_forcing_ratio = max(0.0, 1.0 - epoch / num_epochs)  # ⬅️ Gradual decay

        loop = tqdm(train_loader, desc=f"[Epoch {epoch}/{num_epochs}] Train", leave=False)
        for batch_x, batch_y in loop:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            # -------------------------------
            # 입력 데이터에 미세한 노이즈 추가 (regularization)
            # 참고 논문: "Understanding Deep Learning Requires Rethinking Generalization" (arXiv:1611.03530)
            # -------------------------------
            with torch.no_grad():
                feature_noise_std = torch.tensor([4e-4] * batch_x.size(-1), device=batch_x.device)
                noise = torch.randn_like(batch_x) * feature_noise_std
                batch_x += noise

            optimizer.zero_grad()

            # -------------------------------
            # Scheduled Sampling 기반 Autoregressive Training
            # (모델의 예측값을 다음 입력에 섞어 넣음)
            # 참고 논문: "Scheduled Sampling for Sequence Prediction with Recurrent Neural Networks" (arXiv:1506.03099)
            # -------------------------------
            B, T, F = batch_x.shape
            output_len, output_dim = 1, batch_y.shape[-1]

            seq_in = batch_x.clone()
            all_preds = []
            
            for t in range(output_len):  # currently 1 step
                out = model(seq_in)                  # shape: (B, 1, 4)
                pred = out[:, -1, :]                 # (B, 4)
                all_preds.append(pred.unsqueeze(1))  

                use_pred = torch.rand(B, device=device) > teacher_forcing_ratio

                # Only used if output_len > 1 in the future
                if t < output_len - 1:
                    pred_input = pred.detach()
                    true_input = batch_y[:, t, :]     # (B, 4)

                    mixed_input = torch.where(use_pred.unsqueeze(1), pred_input, true_input)  # (B, 4)

                    # Build new input feature for next time
                    dest_lat, dest_lon = batch_x[:, -1, 4], batch_x[:, -1, 5]
                    delta_lat = dest_lat - mixed_input[:, 0]
                    delta_lon = dest_lon - mixed_input[:, 1]
                    dist = torch.sqrt(delta_lat**2 + delta_lon**2)

                    next_input = torch.cat([
                        mixed_input,                                 # 4
                        dest_lat.unsqueeze(1),                       # 1
                        dest_lon.unsqueeze(1),                       # 1
                        dist.unsqueeze(1),                           # 1
                    ], dim=1)                                        # → (B, 7)

                    seq_in = torch.cat([seq_in[:, 1:, :], next_input.unsqueeze(1)], dim=1)

            # Final loss
            preds = torch.cat(all_preds, dim=1)  # (B, 1, 4)
            loss = criterion(preds, batch_y)
            loss.backward()

            # -------------------------------
            # Gradient Clipping (기울기 폭주 방지)
            # 참고 논문: "Attention Is All You Need" (arXiv:1706.03762)
            # -------------------------------
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_loss = total_loss / len(train_loader)
        print(f"[Epoch {epoch}/{num_epochs}] Train Loss: {avg_loss:.6f}")

        # -------------------------------
        # 검증 (Validation) 단계 - Scheduled Sampling 없이 직접 예측만 수행
        # -------------------------------
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for val_x, val_y in val_loader:
                val_x, val_y = val_x.to(device), val_y.to(device)
                val_out = model(val_x)
                val_loss = criterion(val_out, val_y)
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"           ↳ Val Loss: {avg_val_loss:.6f}")

        # -------------------------------
        # 베스트 모델 저장 (Validation loss 기준)
        # -------------------------------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model, save_path)
            print(f"           ↳ ✅ Best model saved (Val Loss: {best_val_loss:.6f})")

        scheduler.step()

for i in range(10):
    model_name = f"model_{i}.pth"
    
    # 모델 생성
    model = TransformerPredictor(input_size=7, output_size=4)
    
    # numpy → torch tensor로 변환
    input_tensor = torch.tensor(input_seqs, dtype=torch.float32)
    output_tensor = torch.tensor(output_seqs, dtype=torch.float32)
    # 학습
    train_transformer_model(model, (input_tensor, output_tensor), num_epochs=100, device='cuda' if torch.cuda.is_available() else 'cpu')

    torch.save(model, model_name)

# 5. Autoregressive Predict Pipeline

In [ ]:
import torch
import numpy as np

def predict_autoregressive(model, initial_seq, dest_lat, dest_lon, scaler, max_steps, distance_threshold=0.1, device='cuda'):
    model.eval()
    input_seq = initial_seq.clone().to(device)
    all_preds_mu = []

    # 목적지 좌표 정규화
    dest_scaled = scaler.transform([[dest_lat, dest_lon, 0, 0]])[0]
    dest_lat_scaled, dest_lon_scaled = dest_scaled[0], dest_scaled[1]
    
    # 초기 상태 시퀀스도 넣어주기
    for step in range(max_steps):
        with torch.no_grad():
            mu = model(input_seq)
            pred_mu_np = mu.squeeze(1).cpu().numpy().flatten()  # (4,)
            
        all_preds_mu.append(pred_mu_np)

        # 예측된 lat/lon이 목적지에 도달했는지 판정
        pred_mu_denorm = scaler.inverse_transform(pred_mu_np.reshape(1, -1))[0]
        pred_lat, pred_lon = pred_mu_denorm[:2]

        if step % 10 == 0:
            print(f"[Step {step+1}] Predicted: ({pred_lat:.5f}, {pred_lon:.5f}) | "
                  f"Target: ({dest_lat:.5f}, {dest_lon:.5f}) | "
                  f"ΔLat: {abs(pred_lat - dest_lat):.5f}, ΔLon: {abs(pred_lon - dest_lon):.5f}")

        if abs(pred_lat - dest_lat) < distance_threshold and abs(pred_lon - dest_lon) < distance_threshold:
            print(f"🚢 목적지 도달 - Step: {step + 1}, {int(step/12)} 시간 {int((step%12)*5)} 분 소요")
            break

        # 시퀀스 슬라이딩
        input_seq_np = input_seq.squeeze(0).cpu().numpy()  # shape: (T, 11)

        # 다음 입력 feature 구성
        next_input_features = pred_mu_denorm  # [lat, lon, sog, cog]
        next_input_features_scaled = scaler.transform(next_input_features.reshape(1, -1))[0]

        # Δlat, Δlon
        delta_lat = dest_lat - pred_lat
        delta_lon = dest_lon - pred_lon

        # 거리
        distance = np.sqrt(delta_lat ** 2 + delta_lon ** 2)

        # 총 7차원 next input vector
        next_input_array = np.concatenate([
            next_input_features_scaled,            # 4개
            [dest_lat_scaled, dest_lon_scaled],    # 2개
            [distance],                            # 1개
        ])

        # 시퀀스 업데이트
        new_seq = np.vstack([input_seq_np[1:], next_input_array.reshape(1, -1)])  # shape: (T, 11)
        input_seq = torch.tensor(new_seq, dtype=torch.float32).unsqueeze(0).to(device)

    return np.stack(all_preds_mu)

In [ ]:
from math import atan2, sqrt, degrees
from math import pi, sin, cos
# 초기 시퀀스
# 예시: 특정 CSV 파일에서 인천항 출발 경로 하나 로드
pre = AISPreprocessor(data_dir='rou', input_seq_len=12, output_seq_len=1)
#dp_route_1_202002 #ij_route_1_202002 #ud_route_1_202002 #yu_route_1_202003
df = pd.read_csv('rou/yu_route_1_202003.csv', encoding='cp949', parse_dates=['일시'])
df = pre._preprocess_single_file(df)
initial_seq = df.iloc[:12]

# MinMaxScaler 수동 설정 ----------------------------------------------------------
# 정규화 범위 설정
lat_range = (33.0, 38.0)
lon_range = (124.0, 132.0)
sog_range = (0.0, 100.0)
cog_range = (0.0, 360.0)

input_scaler = MinMaxScaler()
input_scaler.min_ = np.array([
    -lat_range[0] / (lat_range[1] - lat_range[0]),
    -lon_range[0] / (lon_range[1] - lon_range[0]),
    -sog_range[0] / (sog_range[1] - sog_range[0]),
    -cog_range[0] / (cog_range[1] - cog_range[0])
])
input_scaler.scale_ = np.array([
    1 / (lat_range[1] - lat_range[0]),
    1 / (lon_range[1] - lon_range[0]),
    1 / (sog_range[1] - sog_range[0]),
    1 / (cog_range[1] - cog_range[0])
])
input_scaler.feature_names_in_ = np.array(['위도', '경도', 'SOG', 'COG'])
# --------------------------------------------------------------------------------

# 목적지 좌표 설정 - 포항
pohang = (36.0320, 129.3884)
donghae = (37.5340, 129.1161)
ulsan = (35.4985, 129.3850)
mokpo  = (34.7925, 126.3814)
jeju = (33.5136, 126.5230)

dest_lat = ulsan[0]
dest_lon = ulsan[1]

# 입력 시퀀스 생성
test_input_seq = []

for i, (_, row) in enumerate(initial_seq.iterrows()):
    # 1. 정규화된 위도, 경도, SOG, COG
    scaled = input_scaler.transform([[row['위도'], row['경도'], row['SOG'], row['COG']]])[0]
    
    # 2. 목적지 위도, 경도 정규화
    dest_scaled = input_scaler.transform([[dest_lat, dest_lon, 0, 0]])[0]
    dest_lat_scaled = dest_scaled[0]
    dest_lon_scaled = dest_scaled[1]

    # 3. Δlat, Δlon (정규화 x)
    delta_lat = dest_lat - row['위도']
    delta_lon = dest_lon - row['경도']
    
    # 4. 거리
    distance = np.sqrt(delta_lat ** 2 + delta_lon ** 2)

    # 최종 특성 벡터
    input_row = list(scaled) + [dest_lat_scaled, dest_lon_scaled, distance]
    test_input_seq.append(input_row)

# 7. 텐서로 변환: (1, seq_len, 7)
test_input_seq = torch.tensor([test_input_seq], dtype=torch.float32)

In [ ]:
def inverse_transform_preds(preds, scaler):
    """
    역정규화를 수행하여 원래의 값으로 변환
    preds: 예측된 값들 (numpy 배열), shape: (steps, output_size)
    scaler: 학습에 사용된 MinMaxScaler
    """
    # 위도, 경도, SOG, COG 값만 역정규화
    preds_unscaled = preds.copy()  # 예측된 값을 복사

    # 위도, 경도, SOG, COG를 역정규화
    preds_unscaled[:, :4] = scaler.inverse_transform(preds_unscaled[:, :4])  # 역정규화
    return preds_unscaled

import folium
from folium.plugins import AntPath

def visualize_route(initial_seq, preds_inverse, dest_lat, dest_lon, scaler):
    """
    예측된 경로를 시각화하는 함수
    initial_seq: 초기 입력 시퀀스 (numpy 배열), shape: (10, 6)
    preds_inverse: 역정규화된 예측 결과, shape: (steps, 4)
    dest_lat, dest_lon: 목적지 좌표
    """
    # 초기 위치
    start = initial_seq[0, 0][:4]
    start = scaler.inverse_transform([start])
    start_lat = start[0][0] 
    start_lon = start[0][1]
    # 지도 생성 (출발지와 목적지가 모두 보이도록 설정)
    route_map = folium.Map(location=[start_lat, start_lon], zoom_start=6)

    # 시작점, 목적지 마커 추가
    folium.Marker([start_lat, start_lon], tooltip='Start', icon=folium.Icon(color='green')).add_to(route_map)
    folium.Marker([dest_lat, dest_lon], tooltip='Destination', icon=folium.Icon(color='red')).add_to(route_map)

    # 예측 경로
    route_coords = [[lat, lon] for lat, lon in preds_inverse[:, :2]]  # 위도, 경도만 사용
    # 예측 경로를 PolyLine으로 시각화
    folium.PolyLine(route_coords, color='blue', weight=3, tooltip="Predicted Route").add_to(route_map)

    # 예측 경로에 애니메이션 효과 추가
    AntPath(route_coords).add_to(route_map)

    return route_map    

# 예측 결과를 역정규화 후 시각화하는 전체 코드
def predict_and_visualize(model, initial_seq, dest_lat, dest_lon, scaler, max_steps=150, distance_threshold=0.05):
    preds = predict_autoregressive(model, initial_seq, dest_lat, dest_lon, scaler, max_steps, distance_threshold)
    #preds = predict_autoregressive_mc_dropout(model, initial_seq, dest_lat, dest_lon, scaler, max_steps, distance_threshold, num_mc_samples=10) 
    #preds = preds.squeeze(1)
    preds_inverse = inverse_transform_preds(preds, scaler)
    # 예측 경로 시각화
    route_map = visualize_route(initial_seq.numpy(), preds_inverse, dest_lat, dest_lon, scaler)

    return route_map, preds_inverse

# 기 시퀀스와 목적지 좌표로 예측 및 시각화
route_map, predictions = predict_and_visualize(model, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map.save('predicted_route_map_v2.html')

# 6. Model 성능 평가

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from folium.plugins import AntPath
from folium import Map, PolyLine
from branca.element import Template, MacroElement
from branca.element import Element

def plot_routes_on_map(actual_coords, predicted_coords, metrics):
    # 지도 초기화
    m = folium.Map(location=actual_coords[0], zoom_start=8)

    # 실제 경로 (파란색 실선)
    folium.PolyLine(
        actual_coords, color='blue', weight=5, opacity=0.7, tooltip='Actual Route'
    ).add_to(m)

    # 예측 경로 (빨간색 점선)
    folium.PolyLine(
        predicted_coords, color='red', weight=3, opacity=0.7, tooltip='Predicted Route', dash_array='10'
    ).add_to(m)
     # 실제 경로 점 표시 (파란색)
    for lat, lon in actual_coords:
        folium.CircleMarker(
            location=(lat, lon),
            radius=4,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=f"Actual: ({lat:.5f}, {lon:.5f})"
        ).add_to(m)

    # 예측 경로 점 표시 (빨간색)
    for lat, lon in predicted_coords:
        folium.CircleMarker(
            location=(lat, lon),
            radius=4,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.7,
            popup=f"Predicted: ({lat:.5f}, {lon:.5f})"
        ).add_to(m)
    # 범례 + 평가 지표 HTML
    legend_html = f"""
    <div style="
        position: fixed;
        bottom: 50px;
        left: 50px;
        z-index: 9999;
        background-color: white;
        border: 2px solid grey;
        border-radius: 5px;
        padding: 10px;
        font-size: 14px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
        line-height: 1.6;
    ">
      <b>🗺 경로 범례</b><br>
      <i style="background:blue; width:10px; height:10px; display:inline-block;"></i> 실제 경로<br>
      <i style="width:20px; height:0px; border-top: 3px dashed red; display:inline-block;"></i> 예측 경로
      <hr style="margin:6px 0;">
      <b>평가 지표</b><br>
      Haversine 평균 거리: {metrics['haversine_km']:.3f} km<br>
      유사도: {metrics['similarity_percent']:.2f} %<br>
      RMSE: {metrics['rmse']:.5f}<br>
      MSE: {metrics['mse']:.5f}
    </div>
    """

    # HTML 요소를 지도에 추가
    m.get_root().html.add_child(Element(legend_html))

    return m


temp_df = df.iloc[12:]
values = temp_df[['위도', '경도']].values

actual_coords = values
predicted_coords = predictions[:,:2]

from haversine import haversine

def haversine_average_distance(actual, predicted):
    min_len = min(len(actual), len(predicted))
    total_distance = sum(haversine(a, p) for a, p in zip(actual[:min_len], predicted[:min_len]))
    return total_distance / min_len  # km


import numpy as np
from sklearn.metrics import mean_squared_error

def coord_rmse_mse(actual, predicted):
    min_len = min(len(actual), len(predicted))
    actual_np = np.array(actual[:min_len])
    pred_np = np.array(predicted[:min_len])

    mse = mean_squared_error(actual_np, pred_np)
    rmse = np.sqrt(mse)
    return rmse, mse


def evaluate_route(actual_coords, predicted_coords):
    # Haversine 평균 거리
    haversine_avg = haversine_average_distance(actual_coords, predicted_coords)

    # Haversine 기반 퍼센트 유사도 (100km 이상은 0%)
    similarity = max(0, 100 * (1 - haversine_avg / 100))

    # 위도/경도 기준 RMSE & MSE
    rmse, mse = coord_rmse_mse(actual_coords, predicted_coords)

    print(f"📍 평균 거리 (Haversine): {haversine_avg:.3f} km")
    print(f"✅ 유사도: {similarity:.2f}%")
    print(f"📉 RMSE (위도+경도 좌표): {rmse:.6f}")
    print(f"📉 MSE  (위도+경도 좌표): {mse:.6f}")

    return {
        'haversine_km': haversine_avg,
        'similarity_percent': similarity,
        'rmse': rmse,
        'mse': mse,
    }

    
metrics = evaluate_route(actual_coords, predicted_coords)
m = plot_routes_on_map(actual_coords, predicted_coords, metrics)
m.save('ph_test.html')
print(len(actual_coords), len(predicted_coords)) #8

# 비교모델 1: LSTM

In [ ]:
import torch
import torch.nn as nn

class LSTMForecast(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=128, num_layers=2, dropout=0.1, output_dim=4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (B, T, input_dim)
        out, _ = self.lstm(x)  # out: (B, T, hidden_dim)
        out = out[:, -1, :]    # 마지막 시점의 hidden state 사용
        out = self.fc(out)     # (B, output_dim)
        return out.unsqueeze(1)  # (B, 1, output_dim) -> [lat, lon, SOG, COG]

# 모델 생성
model = LSTMForecast(input_dim=7, output_dim=4)

# numpy → torch tensor로 변환
input_tensor = torch.tensor(input_seqs, dtype=torch.float32)
output_tensor = torch.tensor(output_seqs, dtype=torch.float32)

# 학습
train_transformer_model(model, (input_tensor, output_tensor), num_epochs=100, device='cuda' if torch.cuda.is_available() else 'cpu')

# 모델 저장
torch.save(model, "lstm_model.pth")
#model = torch.load("lstm_model.pth", weights_only=False)

# 비교모델 2: GRU

In [ ]:
import torch
import torch.nn as nn

class GRUForecast(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=128, num_layers=2, dropout=0.1, output_dim=4):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (B, T, input_dim)
        out, _ = self.gru(x)   # out: (B, T, hidden_dim)
        out = out[:, -1, :]    # 마지막 시점 hidden state
        out = self.fc(out)     # (B, output_dim)
        return out.unsqueeze(1)  # (B, 1, output_dim)

# 모델 생성
model = GRUForecast(input_dim=7, output_dim=4)

# numpy → torch tensor로 변환
input_tensor = torch.tensor(input_seqs, dtype=torch.float32)
output_tensor = torch.tensor(output_seqs, dtype=torch.float32)

# 학습
train_transformer_model(model, (input_tensor, output_tensor), num_epochs=100, device='cuda' if torch.cuda.is_available() else 'cpu')

# 모델 저장
torch.save(model, "gru_model.pth")

# 각 모델별 성능 평가 지표(그래프)

In [ ]:
model_trans = torch.load("1pick.pth", weights_only=False)
model_lstm = torch.load("lstm_model.pth", weights_only=False)
model_gru = torch.load("gru_model.pth", weights_only=False)

In [ ]:
pre = AISPreprocessor(data_dir='rou', input_seq_len=12, output_seq_len=1)
#dp_route_1_202002
#ij_route_1_202002
#ud_route_1_202002
#yu_route_1_202003
df = pd.read_csv('rou/ij_route_1_202002.csv', encoding='cp949', parse_dates=['일시'])
df = pre._preprocess_single_file(df)
initial_seq = df.iloc[:12]

# 목적지 좌표 설정 - 포항
pohang = (36.0320, 129.3884)
donghae = (37.5340, 129.1161)
ulsan = (35.4985, 129.3850)
mokpo  = (34.7925, 126.3814)
jeju = (33.5136, 126.5230)

dest_lat = jeju[0]
dest_lon = jeju[1]

# 입력 시퀀스 생성
test_input_seq = []

for i, (_, row) in enumerate(initial_seq.iterrows()):
    # 1. 정규화된 위도, 경도, SOG, COG
    scaled = input_scaler.transform([[row['위도'], row['경도'], row['SOG'], row['COG']]])[0]
    
    # 2. 목적지 위도, 경도 정규화
    dest_scaled = input_scaler.transform([[dest_lat, dest_lon, 0, 0]])[0]
    dest_lat_scaled = dest_scaled[0]
    dest_lon_scaled = dest_scaled[1]

    # 3. Δlat, Δlon (정규화 x)
    delta_lat = dest_lat - row['위도']
    delta_lon = dest_lon - row['경도']
    
    # 4. 거리
    distance = np.sqrt(delta_lat ** 2 + delta_lon ** 2)

    # 최종 특성 벡터
    input_row = list(scaled) + [dest_lat_scaled, dest_lon_scaled, distance]
    test_input_seq.append(input_row)

# 7. 텐서로 변환: (1, seq_len, 7)
test_input_seq = torch.tensor([test_input_seq], dtype=torch.float32)

In [ ]:
route_map, dpredictions_trans = predict_and_visualize(model_trans, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map, dpredictions_lstm = predict_and_visualize(model_lstm, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map, dpredictions_gru = predict_and_visualize(model_gru, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
temp_df = df.iloc[12:]
values = temp_df[['위도', '경도']].values
actual_coords = values
dactual_coords = actual_coords

In [ ]:
route_map, ipredictions_trans = predict_and_visualize(model_trans, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map, ipredictions_lstm = predict_and_visualize(model_lstm, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map, ipredictions_gru = predict_and_visualize(model_gru, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
temp_df = df.iloc[12:]
values = temp_df[['위도', '경도']].values
actual_coords = values
iactual_coords = actual_coords

In [ ]:
route_map, upredictions_trans = predict_and_visualize(model_trans, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map, upredictions_lstm = predict_and_visualize(model_lstm, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map, upredictions_gru = predict_and_visualize(model_gru, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
temp_df = df.iloc[12:]
values = temp_df[['위도', '경도']].values
actual_coords = values
uactual_coords = actual_coords

In [ ]:
route_map, ypredictions_trans = predict_and_visualize(model_trans, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map, ypredictions_lstm = predict_and_visualize(model_lstm, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map, ypredictions_gru = predict_and_visualize(model_gru, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
temp_df = df.iloc[12:]
values = temp_df[['위도', '경도']].values
actual_coords = values
yactual_coords = actual_coords 

In [ ]:
dpred_trans = dpredictions_trans
dpred_lstm = dpredictions_lstm
dpred_gru  = dpredictions_gru
dactual_d = dactual_coords

ipred_trans = ipredictions_trans
ipred_lstm = ipredictions_lstm
ipred_gru  = ipredictions_gru
iactual_d = iactual_coords

upred_trans = upredictions_trans
upred_lstm = upredictions_lstm
upred_gru  = upredictions_gru
uactual_d = uactual_coords

ypred_trans = ypredictions_trans
ypred_lstm = ypredictions_lstm
ypred_gru  = ypredictions_gru
yactual_d = yactual_coords

import matplotlib.pyplot as plt
import seaborn as sns

# 스타일과 팔레트 설정
sns.set_style("whitegrid")
palette = sns.color_palette("Set2", 4)  # Ground Truth + 3 models

results = {
    "d 항로": (dactual_d, dpred_trans, dpred_lstm, dpred_gru),
    "i 항로": (iactual_d, ipred_trans, ipred_lstm, ipred_gru),
    "u 항로": (uactual_d, upred_trans, upred_lstm, upred_gru),
    "y 항로": (yactual_d, ypred_trans, ypred_lstm, ypred_gru),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
num = 1
for ax, (route_name, (actual, pred_trans, pred_lstm, pred_gru)) in zip(axes.flat, results.items()):
    ax.plot(actual[:,1], actual[:,0], color=palette[0], linewidth=3, label="Ground Truth")
    ax.plot(pred_trans[:,1], pred_trans[:,0], color=palette[1], linestyle="--", linewidth=3, label="Transformer(Ours)")
    ax.plot(pred_lstm[:,1], pred_lstm[:,0], color=palette[2], linestyle="--", label="LSTM")
    ax.plot(pred_gru[:,1], pred_gru[:,0], color=palette[3], linestyle="--", label="GRU")

    ax.set_title(f"Trajectory Comparison - Case {num}", fontsize=13)
    num += 1
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("trajectory_comparison_4routes_seaborn.png", dpi=300)
plt.show()